# 學生學習與作業版｜任務 10：預測時間與資料洩漏

**姓名：**伍亭妤　**學號／組別：**C114118128　**完成日期：**2026/11/25

**學習目標：** 辨認預測時不可用的欄位，區分標籤、特徵與事後資訊。

本單元與教師版「任務 10」相同標題、資料、教學順序與範例；除了明確標示的作業與填空，其餘教師版教材內容保留。對照重點：程式填空對應教師版「10.3 直接看答案」。

### 如何完成
1. 先閱讀觀念與執行資料準備；各單元可以獨立啟動，不需要依賴上一單元的變數。
2. **💻 P 作業**：在指定程式 cell 補上 `____`，可先用提供的 Vibe Coding 提示詞詢問 AI。補完再執行，並執行後面的驗證 cell。未填時出現 `NameError` 是預期現象，先完成填空再繼續。
3. **📝 Q 作業**：查詢 AI、比較教材內容，填寫緊接的文字作答區；請記錄實際提示詞、自己的解釋與查證方式。
4. **🎯 M 小任務**：在任務 4、8、12、16 將前面幾單元整合成可執行的成果。不要只交 AI 回覆，要交實際程式、輸出與自己的解釋。
5. 把 Notebook 存到自己的 Google 雲端硬碟，完成後下載／依教師指示繳交 `.ipynb`；小任務保留輸出、文字與 AI 使用紀錄。

**案例限制：**本資料是公式生成的 60 筆模擬紀錄，不能推論真實學生或因果。缺失與 15 小時值在計分後加入。預測時欄位可得性是假設；全部資料已在前半段探索，後續切出的測試集僅示範流程，不能視為完全獨立的新資料。

**完成檢核：**本單元有 1 項程式填空、2 項 AI 查詢作答；若是任務 4、8、12、16，另外有 1 項整合小任務。請保留自己的輸出、回答和驗證紀錄。


## 共用資料準備
以下在記憶體重建與既有教材相同的資料。執行後可以直接開始本單元；不會覆寫 CSV。


In [2]:
# 匯入套件：numpy 建立隨機資料，pandas 管理表格，pyplot 繪圖
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# 固定種子，並維持呼叫順序，以重現原教材的 60 筆模擬資料
rng = np.random.default_rng(42)
n = 60
departments = ["資訊管理", "企業管理", "金融管理"]
# 每個字典鍵是一個欄位；對應的值是 60 名學生的資料
data = {
    "student_id": range(1, n + 1),
    "department": rng.choice(departments, n),
    "sleep_hours": rng.normal(6.7, 1.1, n).round(1),
    "screen_time_hours": rng.normal(5.0, 1.5, n).clip(1, 10).round(1),
    "study_hours": rng.normal(2.5, 1.0, n).clip(0.5, 6).round(1),
    "attendance_rate": rng.normal(88, 8, n).clip(60, 100).round(0),
}
df = pd.DataFrame(data)
# 成績依設定公式與隨機誤差建立；關係來自模擬規則，不能視為因果發現
noise = rng.normal(0, 5, n)
score = (50 + 3 * df["study_hours"] + 2 * df["sleep_hours"]
         - 1.2 * df["screen_time_hours"] + 0.15 * df["attendance_rate"] + noise)
df["quiz_score"] = score.clip(0, 100).round(0)
# 故意在計分後加入資料品質問題；無法從表格還原計分時原始值
df.loc[[5, 17], "sleep_hours"] = np.nan
df.loc[8, "screen_time_hours"] = 15.0
print("模擬資料形狀（列, 欄）：", df.shape)


模擬資料形狀（列, 欄）： (60, 7)


## 10.1 欄位可用性
**觀念說明：**在考前預測，考後才知道的 `quiz_score` 不可作輸入；`needs_support` 本身是標籤，也不能當特徵。`student_id` 是識別碼通常排除。其他行為欄位的考前可得性僅為教學假設。


In [3]:
# 建立教師可投影的欄位檢查表
availability = pd.DataFrame({
 "field": ["student_id", "department", "study_hours", "sleep_hours", "screen_time_hours", "attendance_rate", "quiz_score", "needs_support"],
 "decision": ["排除", "假設可用", "假設可用", "假設可用", "假設可用", "假設可用", "排除", "標籤，不可作輸入"],
 "reason": ["識別碼", "需確認可得時間", "需確認可得時間", "需確認可得時間", "需確認可得時間", "需確認可得時間", "考後才知道，且用來建標籤", "答案本身"]})
display(availability)


,field,decision,reason
0,student_id,排除,識別碼
1,department,假設可用,需確認可得時間
2,study_hours,假設可用,需確認可得時間
3,sleep_hours,假設可用,需確認可得時間
4,screen_time_hours,假設可用,需確認可得時間
5,attendance_rate,假設可用,需確認可得時間
6,quiz_score,排除,考後才知道，且用來建標籤
7,needs_support,標籤，不可作輸入,答案本身


## 📝 學生作業 10-Q1｜AI 查詢與作答：退訂洩漏
**任務：**預測下個月退訂時，為何退訂後才有的取消日期不可作輸入？

**可使用的 AI 提示詞：**
```text
我是 Python 與資料分析初學者。請用時間線說明客戶取消日期造成目標洩漏，並舉例可考慮的預測前欄位。
請使用繁體中文、台灣常用名詞，分步解釋，並提出我可以自行驗證的一個例子；不要假設我的程式輸出。
```
先閱讀教材，再詢問 AI。請記錄工具名稱、實際提示詞；若 AI 提供資料來源，記錄可查證的網址／文件名稱與日期，並自行檢查，無法查證時寫「尚未查證」。不要提供真實個資。


### ✍️ 作業 10-Q1 專用作答區：請雙擊此文字儲存格填寫
- AI 工具及實際提示詞：Gemini，提示詞：你好我是 Python 與資料分析初學者。請用時間線說明客戶取消日期造成目標洩漏，並舉例可考慮的預測前欄位。請分步解釋，並提出我可以自行驗證的一個例子；不要假設我的程式輸出。
- AI 回答重點（請自行整理）：目標洩漏是指在訓練模型時，不小心放入了「只有在未來才會出現的資訊」，導致模型在訓練時作弊拿到高分，但真正上線預測未來時卻完全無法使用。例子：若在10月1日預測客戶當月是否流失，該客戶在10月15日才取消訂閱並產生cancel_date。11月事後撈取歷史資料訓練時，若未剔除該欄位，模型會直接學會「有cancel_date = 一定流失」，但 10 月 1 日預測當下該欄位其實是空值。
- 我的理解與具體例子：就像是在期末考前答案卻先一步被流出來，造成目標洩漏的概念。例如：如果要預測用戶下個月會不會購買付費會員，卻不小心把電子發票開立日期或會員等級變成 VIP當成輸入特徵，這些欄位都是結帳後才會產生的，在預測當下根本不存在，這就是目標洩漏。
- 查證來源／方式及日期（無法查證請註明）：https://scikit-learn.org/stable/common_pitfalls.html
  2026/11/25
- 我檢查或修正 AI 回答的地方：無


## 10.2 退訂案例
如果在本月底預測下個月是否退訂，且取消日期只有退訂後才產生，`cancellation_date` 就不能作為本月底預測的輸入；最近登入等欄位也必須確認截點與統計視窗。


In [4]:
# 建立獨立小表，對照預測當下能否使用
churn = pd.DataFrame({"customer_id": ["A", "B", "C"],
 "logins_before_cutoff": [3, 10, 1], "cancellation_date": ["2026-02-05", None, "2026-02-12"],
 "churn_next_month": [1, 0, 1]})
display(churn)
print("可以考慮的候選：", ["logins_before_cutoff"])
print("排除的事後欄位：", ["cancellation_date"])


,customer_id,logins_before_cutoff,cancellation_date,churn_next_month
0,A,3,2026-02-05,1
1,B,10,NaN,0
2,C,1,2026-02-12,1


可以考慮的候選： ['logins_before_cutoff']
排除的事後欄位： ['cancellation_date']


## 📝 學生作業 10-Q2｜AI 查詢與作答：欄位時間
**任務：**看見欄位名稱後，還要確認哪些收集或計算時間，才能判斷是否可用？

**可使用的 AI 提示詞：**
```text
我是 Python 與資料分析初學者。請說明截點、統計窗口、欄位可得時間如何影響預測特徵是否合法使用。
請使用繁體中文、台灣常用名詞，分步解釋，並提出我可以自行驗證的一個例子；不要假設我的程式輸出。
```
先閱讀教材，再詢問 AI。請記錄工具名稱、實際提示詞；若 AI 提供資料來源，記錄可查證的網址／文件名稱與日期，並自行檢查，無法查證時寫「尚未查證」。不要提供真實個資。


### ✍️ 作業 10-Q2 專用作答區：請雙擊此文字儲存格填寫
- AI 工具及實際提示詞：Gemini，提示詞：你好我是 Python 與資料分析初學者。請說明截點、統計窗口、欄位可得時間如何影響預測特徵是否合法使用。請分步解釋，並提出我可以自行驗證的一個例子；不要假設我的程式輸出。
- AI 回答重點（請自行整理）：截點：模型進行預測的時間基準線。統計窗口：用來計算歷史行為特徵的時間範圍。欄位可得時間：資料實際寫入資料庫、可供系統查詢的時間。判斷特徵合法：必須同時滿足「事件發生在統計窗口內」且「欄位可得時間早於或等於截點」，否則即便事件發生在過去，若系統在截點後才入帳更新，仍會造成時間穿越洩漏。
- 我的理解與具體例子：合法必須要截點不超過欄位可的時間，且事情需要發生在統計窗口內。例如：在10月1日是截點，預測客戶當月是否會退貨，統計窗口為過去30天的商品評分。若客戶在9月30日填寫負評，但系統因人工審核延遲到10月2日才公開寫入資料庫（欄位可得時間），該負評在10月1日就屬於不可用的未來資訊。
- 查證來源／方式及日期（無法查證請註明）：尚未查證
- 我檢查或修正 AI 回答的地方：AI給的邊界符號是小於等於但他之前給我的小實驗是只有小於，所以到時候判斷應該要統一一下，否則前後邏輯會不一樣。


## 10.3 直接看答案的危險示範
下列程式不是合格模型：它直接使用事後分數計算答案。高正確率只能證明標籤規則被重複，不能證明可以在考前預測。


## 💻 學生作業 10-P1｜Vibe Coding 程式填空
**對應教師版：**任務 10「10.3 直接看答案」的示範程式。

**任務：**刻意利用考後 quiz_score，依低於 75 產生 leaked_answer，比較與 actual；這是錯誤情境示範，不是可部署模型。

**執行方式：**找到下一個程式儲存格中的 `____`，使用自己撰寫或 AI 建議的程式補齊，保留其他已提供的程式與註解。補完後先執行該 cell，再執行緊接的自動檢查；後續單元內容可能依賴此變數。

**可使用的 Vibe Coding 提示詞：**
```text
我是 Python 與資料分析初學者，正在 Google Colab 使用 pandas、numpy、matplotlib 或 scikit-learn。
我的任務是：刻意利用考後 quiz_score，依低於 75 產生 leaked_answer，比較與 actual；這是錯誤情境示範，不是可部署模型。
請依我貼上的上下文，只協助補上 ____，保留指定變數名稱與後續程式，不修改原始 df。
請解釋每一步的作用，提供繁體中文註解及可手動驗證的方法。
不要虛構執行結果；如需額外資訊，先指出缺少什麼。
```
完成後記錄實際使用的提示詞與自己改動的地方。


In [5]:
# 刻意錯誤示例：事後分數直接提供了標籤資訊
actual = (df["quiz_score"] < 75).astype(int)
leaked_answer = (df["quiz_score"] < 75).astype(int) # TODO：刻意用考後分數重現標籤，示範洩漏
print("偷看答案的表面正確率：", (actual == leaked_answer).mean())
# 這個 100% 沒有實務預測效益：考前尚未取得 quiz_score


偷看答案的表面正確率： 1.0


### ✅ 作業 10-P1｜執行驗證
執行下方程式，並同時檢查上方實際輸出。自動檢查只涵蓋主要條件；如果圖表、說明或資料來源不符任務，仍需人工修正。


In [6]:
assert (leaked_answer == actual).all()
print("檢查通過：事後欄位可直接重現標籤；這正是洩漏警訊。")


檢查通過：事後欄位可直接重現標籤；這正是洩漏警訊。


### ✍️ 作業 10-P1｜我的完成紀錄
- 我實際使用的提示詞（若未使用 AI 請註明）：你好我是 Python 與資料分析初學者，正在 Google Colab 使用 pandas、numpy、matplotlib 或 scikit-learn。我的任務是刻意利用考後 quiz_score，依低於 75 產生 leaked_answer，比較與 actual；這是錯誤情境示範，不是可部署模型。請依我貼上的上下文，只協助補上 ____，保留指定變數名稱與後續程式，不修改原始 df。請解釋每一步的作用，提供繁體中文註解及可手動驗證的方法。不要虛構執行結果；如需額外資訊，先指出缺少什麼。
- 我補上的程式在做什麼：刻意使用「考試結束後才會產生的 quiz_score」來生成預測結果leaked_answer。在真實的預測場景（考試前）根本拿不到這個欄位，因此這是一個典型的目標洩漏（Target Leakage）錯誤示範。
- 執行或驗證遇到的問題與修正：無
- 檢查結果及我如何確認：print("偷看答案的表面正確率：", (actual == leaked_answer).mean()) 後，印出的結果為1.0，因為leaked_answer與actual是用完全相同的考後分數與條件計算出來的。而且10-P1執行驗證中顯示檢查通過：事後欄位可直接重現標籤；這正是洩漏警訊。


## 10.4 Explain-back
請學生用自己的話區分：建立訓練標籤可用已知結果，但模型輸入只能用預測時間點可取得的資料。列出兩個真實專案會追問的欄位時間資訊。

區分：製作訓練資料時，我們是站在事後回顧的上帝視角，因此可以拿未來已經發生的結果來當作答案；但用來做預測的題目，必須嚴格限制在「預測當下（截點）就已經確定存在」的資料，不能偷用未來才知道的資訊。

案例：
1.事件發生時間： 這筆行為或紀錄實際發生的時間點（例如：客戶實際下單或點名簽到的時間），用來確認它是否落在截點前的統計窗口內。

2.資料入帳／更新時間：這筆資料實際寫入資料庫、可被系統查詢到的時間，用來確認在預測當下這筆資料是否真的已經拿得到，避免因延遲寫入而造成時間穿越洩漏。

## 延伸：把時間點寫成資料規格
例如：以 1 月 31 日 23:59 為截點，預測 2 月是否退訂；登入次數的統計時間必須截止於 1 月 31 日。不能僅看欄位名稱就認定一定安全，也要檢查欄位的計算方式。

**學生練習：**為出席率、睡眠時間各寫下蒐集窗口、資料可得時間與一個可能洩漏的情境。

蒐集窗口：(出席率)開學第1週至第九週(例如9/1～10/31 23:59)的課堂點名紀錄。(睡眠時間)期中考前7天穿戴裝置記錄的睡眠時數。

資料可的時間：(出席率)每堂課點名系統同步後。(睡眠時間)學生起床後開啟 App 同步手錶資料的時間。

情境：(出席率)計算時不小心把第九週後的點名也算進出席率。(睡眠時間)學生考前一天晚上忘記連線同步手錶，直到考完試中午才上傳昨晚的睡眠數據

## 本單元學習收束與 AI 使用紀錄

請學生先用自己的話解釋一個關鍵觀念，再指出一個執行結果及驗證方式。記錄：使用的提示詞、AI 回答中檢查或修改的地方、遇到的問題、修正後的證據與本資料限制。

**通用提示詞**：我是 Python 初學者，請逐步協助我完成上面的任務；每段程式加繁體中文註解，說明輸入、處理與輸出，提供兩項可實際執行的驗證，保留原始資料，不要虛構執行結果。
